UNet - SR - v2 - Inference

In [1]:
from fastai.vision.all import *
#from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy
#import matplotlib.pyplot as plt
import os
import pathlib
import torch
#from itertools import *
#from itertools import islice
#from torchvision.utils import * 
#import torchvision.transforms as tfms
#from functools import partial
#import timm
import fastai; 
fastai.__version__

'2.7.15'

In [2]:
torch.__version__

'2.3.1+cu121'

In [3]:
torch.device("cuda")
device="cuda"
torch.cuda.set_device(0)

In [4]:
path = Path.cwd()/'data/'

In [5]:
path_LR = path/'x2/train'

In [7]:
# Set seeds for reproducibility
%run util/set_seeds.ipynb

# run Dataloader
%run util/Diffusion_Dataloader_inference.ipynb

# run notebook to set up feature loss 
%run util/Diffusion_Perception_loss.ipynb

/home/david/miniforge3/envs/fastai/lib/python3.10/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/david/miniforge3/envs/fastai/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
dls_den, dblock = get_dls(bs=32)        

In [16]:
def create_learner():
    return unet_learner(dls_den, bbone, loss_func=loss_func, blur=True, norm_type=NormType.Weight, 
                        self_attention=True, y_range=y_range, pretrained=True, splitter=default_split
                        )

In [19]:
#generate list of models available 
model_list = sorted(get_files(path=Path.cwd(), extensions='.pth', folders='models'))
for i in range(len(model_list)):
    model_list[i]=model_list[i].with_suffix('').name
    print(i, ":", model_list[i]) 

0 : SR_R18_L1


In [11]:
#prediction loop
def save_preds(dl, learn,path_preds):
  "Save away predictions"
  names = dl.dataset.items  
  path_preds=path_preds
  path_preds.mkdir(exist_ok=True)
    
  inp, preds, _,decoded = learn.get_preds(dl=dl, with_input=True, with_decoded=True)

  for i,pred in enumerate(preds):      
      arr = pred.clamp(0,1).permute(1,2,0).numpy()
      arr = arr[:,:,0]*(2**16-1)
      cv2.imwrite(str(path_preds/names[i].name),arr.astype(np.uint16))
  return inp, preds, decoded       # return outputs for analysis

In [22]:
#setting up path to test data and test data itself
path_base = Path.cwd()/'data/x2/test'

In [23]:
loss_func = FeatureLoss(vgg_m, blocks[2:5], [125],[1,1,1,1,1], [1,1,1,1,1])        # dummy

In [24]:
Testfolders=['115017']
#Testfolders=['115017','115825','116726','118225','125525'] # test set in publication
indizes = [0] # select which models to use frome list above
chunk_size = 5000 # set chunk size based on available memory
#inference loop for models
for k,i in enumerate(indizes):
    print('loading model:' +model_list[i])
    learn_den = create_learner()
    learn_den.model.to(device) # move model to gpu
    learn_den.dls.to(device) # move dataloader to gpu
    learn_den.to_fp16() # set to fp16 for inference speedup
    learn_den.model_dir = Path.cwd()/'models' # set path to model weights
    learn_den.load(model_list[i]) # load model weights

    for j in Testfolders:
        name_test = j
        path_test = path_base/name_test 
        test_files = get_image_files(path_test,recurse=False)                     
        print(path_test, len(test_files))
        for k in range(0,len(test_files),chunk_size):
            chunk = test_files[k:k+chunk_size]
            test_dl=learn_den.dls.test_dl(chunk)                      # use test_dl from dataloader              
            name=j+'_'+model_list[i]
            save_preds(test_dl, learn_den,path_base/name)   # run prediction and save outputs

loading model:SR_R18_L1


/home/david/miniforge3/envs/fastai/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
/home/david/miniforge3/envs/fastai/lib/python3.10/site-packages/fastai/learner.py:58: UserWarning: Could not load the optimizer state.
  if with_opt: warn("Could not load the optimizer state.")


/home/david/mnt_anarchyNAS/lohrd/Github/Sharp-diffusion/data/x2/test/115017 24739
